In [1]:
!pip install python-dotenv newspaper3k

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 2.4 MB/s eta 0:00:00 MB/s eta 0:00:01:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13541 sha256=06d6dfd3d6bfa6453a1dc4d6d1ffa4265d7006ab8b67d8cad1d33a0787173792
  Stored in directory: /home/vikas/.cache/pip/wheels/c8/d6/6c/384f58df48c00b9a31d638005143b5b3ac62c3d25fb1447f23
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3342 sha256=5d127fd4a1912c02cbc961c8c405808d9135ed67a0a7c7861088b465553176fe
  Stored in directory: /home/vikas/.cache/pip/wheels/97/02/e7/a1ff1760e12bdbaab0ac824fae5c1bc933e41c4ccd6a8f8edb
  Created wheel for jieba3k: filename=jieba3k-0.35.1-py3-none-any.whl size=7398379 sha256=f9ba88dceddeb96f30d3d2ee84e7fc488dfa1df943b6003adda22935dafecfcf
  Stored in directory: /home/vikas/.

In [2]:
!pip install -U "langchain[anthropic]"

In [12]:
from dotenv import load_dotenv
import os
# news api pub_bff9f5c4768c4229b13429a9b7c67929
load_dotenv(".env")
#check if key is set 
value = os.getenv("ANTHROPIC_API_KEY")
if value: 
    masked = value[:4] + "..." + value[-4:] if len(value) > 8 else "***" 
else: 
    masked = "<empty>"
print(masked)

sk-a...2gAA


In [13]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(
    "claude-sonnet-4-6",
    model_provider="anthropic",
    temperature=1
)

In [14]:
import requests
from newspaper import Article

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.82 Safari/537.36'
}

article_urls = "https://www.artificialintelligence-news.com/2022/01/25/meta-claims-new-ai-supercomputer-will-set-records/"

session = requests.Session()

try:
    response = session.get(article_urls, headers=headers, timeout=10)

    if response.status_code == 200:
        article = Article(article_urls)
        article.download()
        article.parse()

        print(f"Title: {article.title}")
        print(f"Text: {article.text}")

    else:
        print(f"Failed to fetch article at {article_urls}")
except Exception as e:
    print(f"Error occurred while fetching article at {article_urls}: {e}")

Title: Meta claims its new AI supercomputer will set records
Text: Meta (formerly Facebook) has unveiled an AI supercomputer that it claims will be the world’s fastest.

The supercomputer is called the AI Research SuperCluster (RSC) and is yet to be fully complete. However, Meta’s researchers have already begun using it for training large natural language processing (NLP) and computer vision models.

RSC is set to be fully built in mid-2022. Meta says that it will be the fastest in the world once complete and the aim is for it to be capable of training models with trillions of parameters.

“We hope RSC will help us build entirely new AI systems that can, for example, power real-time voice translations to large groups of people, each speaking a different language, so they can seamlessly collaborate on a research project or play an AR game together,” wrote Meta in a blog post.

“Ultimately, the work done with RSC will pave the way toward building technologies for the next major computing

In [18]:
from langchain_core.messages import HumanMessage

# we get the article data from the scraping part
article_title = article.title
article_text = article.text

# prepare template for prompt so it can be used for many aarticles by just changiung the variable
prompt_template = """You are a very good assistant that summarizes online articles.

Here's the article you want to summarize.

==================
Title: {article_title}

{article_text}
==================

Write a summary of the previous article.
"""

prompt = prompt_template.format(article_title=article.title, article_text=article.text)
# response = llm.invoke(prompt)
# print(response.content)

#to behave in a role chat based setting we use HumanMessage , A string only contains text. A HumanMessage contains both:
# the text
# the role (human)
#strcutured format conversation, useful when building applications

messages = [
    HumanMessage(content = prompt)
]
response = llm.invoke(messages)
#inside llm chat conversations are saved like iin  a structured manner
# role: system : system prompt
# role: human : human prompt
# answer
print(response.content)


## Summary: Meta Unveils Record-Breaking AI Supercomputer

Meta (formerly Facebook) has announced the development of a new AI supercomputer called the **AI Research SuperCluster (RSC)**, which it claims will be the world's fastest once fully completed in **mid-2022**.

### Key Highlights:
- **Performance**: RSC is expected to be **20x faster** than Meta's current clusters, **9x faster** at running NVIDIA's NCCL, and **3x faster** at training large-scale NLP workflows. Training a model with tens of billions of parameters would take **3 weeks** instead of 9.
- **Capabilities**: The supercomputer aims to train models with **trillions of parameters**, enabling advanced AI applications such as real-time voice translation and augmented reality experiences.
- **Security & Privacy**: Unlike Meta's previous infrastructure, RSC is built with enhanced security and privacy controls, allowing the use of real-world data from Meta's platforms to improve tasks like **harmful content detection**.
- **M

In [23]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)
#good practice to differentiate system and user prompts 
system_prompt = "You are a very good assistant that summarizes online articles."

#templates are reusable, just pass variables using format useful for multiple/dynamic prompts can be used many times.
human_prompt_template ="""
Here's the article you want to summarize.

==================
Title: {article_title}

{article_text}
==================

Write a summary of the previous article.
"""

# we get the article data from the scraping part
article_title = article.title
article_text = article.text

human_prompt = human_prompt_template.format(article_title=article.title, article_text=article.text)

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=human_prompt)
]

response = llm.invoke(messages)
print(response.content)

## Summary: Meta Unveils Record-Breaking AI Supercomputer

Meta has announced the development of its **AI Research SuperCluster (RSC)**, which it claims will be the world's fastest AI supercomputer upon its completion in **mid-2022**.

### Key Highlights:
- **Performance**: RSC is expected to be **20x faster** than Meta's current V100-based clusters, **9x faster** at running NVIDIA's Collective Communication Library (NCCL), and **3x faster** at training large-scale NLP workflows.
- **Capabilities**: The supercomputer is designed to train models with **trillions of parameters**, reducing training time for large models from nine weeks to just three weeks.
- **Current Use**: Researchers are already using RSC for training large **natural language processing (NLP)** and **computer vision** models.

### Intended Applications:
- Real-time voice translation for large, multilingual groups
- Identifying harmful content on Meta's platforms using real-world data
- Supporting the development of the

In [21]:
#mondern way using chains and ChatPromptTemplate

from langchain_core.prompts import ChatPromptTemplate

#prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful summarizer."),
    ("human", """
        Title: {title}

        {text}

        Summarize this article.
    """)
])

#create chain
chain = prompt | llm

#variables passed during invoke
response = chain.invoke({
    "title": article.title,
    "text": article.text
})

In [22]:
print(response.content)

## Summary: Meta's AI Research SuperCluster (RSC)

Meta has unveiled a new AI supercomputer called the **AI Research SuperCluster (RSC)**, which it claims will be the **world's fastest** once fully completed in mid-2022.

### Key Highlights:
- **Purpose:** Designed to train large NLP and computer vision models with potentially **trillions of parameters**
- **Performance:** Expected to be **20x faster** than Meta's current clusters, **9x faster** at running NCCL, and **3x faster** at training large-scale NLP workflows
- **Practical Impact:** Training a model with tens of billions of parameters would take **3 weeks instead of 9**

### Goals & Applications:
- Real-time voice translation across multiple languages
- Identifying harmful content on Meta's platforms using real-world data
- Supporting development of the **metaverse** and AR experiences

### Notable Feature:
Unlike previous infrastructure, RSC was built with enhanced **security and privacy controls**, allowing Meta to train mode

In [24]:
from langchain_core.prompts import ChatPromptTemplate

#prompt template
prompt2_template = ChatPromptTemplate.from_messages([
    ("system", "),
    ("human", """
        Here's the article you need to summarize.
        
        ==================
        Title: {article_title}
        
        {article_text}
        ==================
        
        Now, provide a summarized version of the article in a bulleted list format.
        """
    )
])

messages = [
    systemMessage(content = "You are an advanced AI assistant that summarizes online articles into bulleted lists in French."),
    humanMessage(content = 

## Meta's AI Research SuperCluster (RSC)

Meta has unveiled a new AI supercomputer called the **AI Research SuperCluster (RSC)**, which it claims will be the **world's fastest** once fully completed in mid-2022.

### Key Highlights:
- **Purpose:** Designed to train large NLP and computer vision models with trillions of parameters
- **Performance:** Expected to be 20x faster than Meta's current clusters, 9x faster at running NVIDIA's NCCL, and 3x faster at training large-scale NLP workflows
- **Training Speed:** Models with tens of billions of parameters can be trained in **3 weeks** instead of 9 weeks

### Goals & Applications:
- Real-time voice translation for large groups
- Identifying harmful content on Meta's platforms using real-world data
- Supporting the development of the **metaverse**

### Notable Feature:
RSC was built with enhanced **security and privacy controls**, allowing Meta to train models using real production data for the first time, rather than relying solely on pub